## Scenario ##

You are an independent auditor for the NYC Campaign Finance program. The City of New York provides matching funds for eligible contributions to candidates, with various ratios depending on the contribution amount (see details here). 

As an auditor, you want to get the details on the amounts that NYC gives to each candidate's campaign through this program. You have extracted relevant data from NYC's Open Data Portal for matching contributions. This dataset was downloaded via an API request as a JSON file named nyc_2001_campaign_finance.json.

The dataset is separated into meta, which contains metadata, and data, which contains the actual campaign finance records. You will need to use the information in meta to interpret the information in data.

Your goal is to extract a list containing:

- The name of a candidate in the 2001 election along with the total payments they received
- The list should be structured as a list of tuples (candidate_name, amount)

The list should have 284 typles, matching the number of candidates.

### Step 1: Open the Dataset ###

- Import the json module
- Open the nyc_campaign_financel.json file using the built-in Python open function
- Load all the data from the file into a Python object with json.load
- Assign the results of json.load to the variable name data

In [2]:
import json

In [3]:
with open('nyc_2001_campaign_finance.json') as f: # open with built-in Python open function
    data = json.load(f) # pass the file to json.load

In [4]:
# Overall structure of this dataset

print(f"The overall data type is {type(data)}")
print(f"The keys are {list(data.keys())}")
print()
print("The value associated with the 'meta' key has metadata, including all of these attributes:")
print(list(data['meta']['view'].keys()))
print()
print(f"The value associated with the 'data' key is a list of {len(data['data'])} records")

The overall data type is <class 'dict'>
The keys are ['meta', 'data']

The value associated with the 'meta' key has metadata, including all of these attributes:
['id', 'name', 'attribution', 'averageRating', 'category', 'createdAt', 'description', 'displayType', 'downloadCount', 'hideFromCatalog', 'hideFromDataJson', 'indexUpdatedAt', 'newBackend', 'numberOfComments', 'oid', 'provenance', 'publicationAppendEnabled', 'publicationDate', 'publicationGroup', 'publicationStage', 'rowClass', 'rowsUpdatedAt', 'rowsUpdatedBy', 'tableId', 'totalTimesRated', 'viewCount', 'viewLastModified', 'viewType', 'columns', 'grants', 'metadata', 'owner', 'query', 'rights', 'tableAuthor', 'tags', 'flags']

The value associated with the 'data' key is a list of 285 records


### Step 2: Find the Column Names ###

We know each record in the data list looks something like this:

In [5]:
data['data'][1]

[2,
 '9D257416-581A-4C42-85CC-B6EAD9DED97F',
 2,
 1315925633,
 '392904',
 1315925633,
 '392904',
 '{\n}',
 '2001',
 'B4',
 'Aboulafia, Sandy',
 '5',
 None,
 '44',
 'P',
 '45410.00',
 '0',
 '0',
 '45410.00']

The keys of some values are clear, but more digging is needed to identify the total payment received.

In [7]:
column_data = data['meta']['view']['columns'] # defining column_data as a list

type(column_data) # verifying data type

list

In [9]:
# exploring first few entries of column_data

column_data[0]

{'id': -1,
 'name': 'sid',
 'dataTypeName': 'meta_data',
 'fieldName': ':sid',
 'position': 0,
 'renderTypeName': 'meta_data',
 'format': {},
 'flags': ['hidden']}

In [14]:
column_data[-3]

{'id': 75768841,
 'name': 'GENERALPAY',
 'dataTypeName': 'number',
 'fieldName': 'generalpay',
 'position': 10,
 'renderTypeName': 'number',
 'tableColumnId': 1518999,
 'width': 220,
 'cachedContents': {'non_null': 284,
  'average': '28753.57394366197',
  'largest': '976545.00',
  'null': 1,
  'top': [{'item': '0', 'count': 20},
   {'item': '75350.00', 'count': 19},
   {'item': '201131.00', 'count': 18},
   {'item': '39760.00', 'count': 17},
   {'item': '57796.00', 'count': 16},
   {'item': '75200.00', 'count': 15},
   {'item': '68234.00', 'count': 14},
   {'item': '5732.00', 'count': 13},
   {'item': '58488.00', 'count': 12},
   {'item': '62184.00', 'count': 11},
   {'item': '44748.00', 'count': 10},
   {'item': '21946.00', 'count': 9},
   {'item': '70500.00', 'count': 8}],
  'smallest': '0',
  'sum': '8166015.00'},
 'format': {}}

In [15]:
# defining column_names to include only the values associated with the name keys
# each dict has the key 'name', so use list comprehension to extract

column_names = [info['name'] for info in column_data]

column_names

['sid',
 'id',
 'position',
 'created_at',
 'created_meta',
 'updated_at',
 'updated_meta',
 'meta',
 'ELECTION',
 'CANDID',
 'CANDNAME',
 'OFFICECD',
 'OFFICEBORO',
 'OFFICEDIST',
 'CANCLASS',
 'PRIMARYPAY',
 'GENERALPAY',
 'RUNOFFPAY',
 'TOTALPAY']

### Step 3: Loop Over Records to Find Names and Payments ###

Data records are contained in data['data'].

To loop over records, we need to determine the indices of candidate names and total payments.

Let name_index be column names of CANDNAME, and total_payments_index be the column names of TOTALPAY.

Print the respective indices of these variables

In [16]:
name_index = column_names.index("CANDNAME")
total_payments_index = column_names.index("TOTALPAY")

print("The candidate name is at index", name_index)
print("The total payment amount is at index", total_payments_index)

The candidate name is at index 10
The total payment amount is at index 18


Now loop over the records in data['data'] and extract the name from name_index and the total payment from total_payments_index. 

Convert the total payment to a float, then make a tuple representing that candidate. Append the tuple to the overall list of results, candidate_total_payments

Remember that the 0th instance is a header and should be skipped.

To verify the loop is correct, print first five and last five records.

In [20]:
candidate_total_payments = []

for record in data['data'][1:]:
    name = record[name_index]
    total_payment = float(record[total_payments_index])
    candidate_total_payments.append((name, total_payment))

print(candidate_total_payments[:5])
print(candidate_total_payments[-5:])

[('Aboulafia, Sandy', 45410.0), ('Adams, Jackie R', 11073.0), ('Addabbo, Joseph P', 149320.0), ('Alamo-Estrada, Agustin', 27400.0), ('Allen, William A', 62990.0)]
[('Wilson, John H', 0.0), ('Wooten, Donald T', 0.0), ('Yassky, David', 150700.0), ('Zapiti, Mike', 12172.0), ('Zett, Lori M', 0.0)]


Which candidates received the most total payments from the city?

In [21]:
sorted(candidate_total_payments, key=lambda x: x[1], reverse=True)[:10]

[('Green, Mark', 4534230.0),
 ('Ferrer, Fernando', 2871933.0),
 ('Hevesi, Alan G', 2641247.0),
 ('Vallone, Peter F', 2458534.0),
 ('Gotbaum, Betsy F', 1625090.0),
 ('Berman, Herbert E', 1576860.0),
 ('DiBrienza, Stephen', 1336655.0),
 ('Stringer, Scott M', 1223721.0),
 ('Markowitz, Marty', 1166294.0),
 ('Thompson, Jr., William C', 1096359.0)]